# 📱 Google Drive Cloud Unzipper (Engine & Android Bridge)
### Extract Gigabyte-sized ZIP / RAR / 7Z files inside Google Drive in seconds without downloading anything to your phone!

**Features:**
- ⚡ **Zero Mobile Data & Zero Phone Storage:** All extraction happens directly inside Google Cloud.
- 🏎️ **Ultra-Fast 7-Zip Engine:** Multi-core extraction for huge 10GB-30GB video archives.
- 📱 **Android App Connected:** Control everything from your native Android phone app.

## 🚀 1. Launch Cloud Worker for Android App
Run this cell once. It will connect your Google Drive and give you a secure link to paste into your Android App.

In [ ]:
#@title 🚀 1-CLICK LAUNCH CLOUD WORKER FOR ANDROID APP
import os, subprocess, time, re
from google.colab import drive

# 1. Mount Google Drive
if not os.path.exists('/content/drive/MyDrive'):
    print("🔄 Mounting Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive is already mounted.")

# 2. Install p7zip and Python backend packages
print("📦 Installing high-speed 7-Zip and API server...")
subprocess.run(["apt-get", "install", "-y", "-qq", "p7zip-full"], check=False)
subprocess.run(["pip", "install", "-q", "fastapi", "uvicorn", "pydantic"], check=False)

# 3. Download cloudflared for free instant HTTPS tunnel (No account needed!)
if not os.path.exists('/content/cloudflared'):
    print("🌐 Setting up Cloudflare tunnel...")
    subprocess.run(["curl", "-s", "-L", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-o", "/content/cloudflared"], check=True)
    subprocess.run(["chmod", "+x", "/content/cloudflared"], check=True)

# 4. Write server.py
server_code = '''import asyncio, os, glob, subprocess, json
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

class ExtractRequest(BaseModel):
    source_folder: str = "GDFlix"
    destination_folder: str = "MOVIES & WEB SERIES INFO"
    exact_file_name: str = ""
    password: str = ""

@app.get("/")
def health():
    return {"status": "online", "drive": os.path.exists("/content/drive/MyDrive")}

@app.post("/api/extract-stream")
async def extract_stream(req: ExtractRequest):
    async def event_generator():
        yield f"data: {json.dumps({'status': 'starting', 'message': '🚀 Initializing Cloud Extraction...'})}\\n\\n"
        drive_base = "/content/drive/MyDrive"
        src_path = os.path.join(drive_base, req.source_folder.strip("/"))
        dest_path = os.path.join(drive_base, req.destination_folder.strip("/"))

        yield f"data: {json.dumps({'status': 'scanning', 'message': f'🔍 Scanning: {src_path}'})}\\n\\n"
        if not os.path.exists(src_path):
            yield f"data: {json.dumps({'status': 'error', 'message': f'❌ Source folder not found: {src_path}'})}\\n\\n"
            return

        target_archive = None
        if req.exact_file_name.strip():
            candidate = os.path.join(src_path, req.exact_file_name.strip())
            if os.path.exists(candidate):
                target_archive = candidate
            else:
                for f in os.listdir(src_path):
                    if req.exact_file_name.strip().lower() in f.lower():
                        target_archive = os.path.join(src_path, f)
                        break

        if not target_archive:
            files = []
            for ext in ('*.zip', '*.rar', '*.7z', '*.tar', '*.tgz'):
                files.extend(glob.glob(os.path.join(src_path, ext)))
                files.extend(glob.glob(os.path.join(src_path, ext.upper())))
            if files:
                files.sort(key=os.path.getmtime, reverse=True)
                target_archive = files[0]

        if not target_archive or not os.path.exists(target_archive):
            yield f"data: {json.dumps({'status': 'error', 'message': f'❌ No archive found in {req.source_folder}'})}\\n\\n"
            return

        os.makedirs(dest_path, exist_ok=True)
        size_gb = os.path.getsize(target_archive) / (1024 ** 3)
        fname = os.path.basename(target_archive)
        yield f"data: {json.dumps({'status': 'progress', 'progress': 10, 'message': f'🎯 Target: {fname} ({size_gb:.2f} GB)'})}\\n\\n"
        yield f"data: {json.dumps({'status': 'progress', 'progress': 20, 'message': f'📂 Extracting to: MyDrive/{req.destination_folder}'})}\\n\\n"

        cmd = ["7z", "x", "-y", f"-o{dest_path}"]
        if req.password: cmd.append(f"-p{req.password}")
        else: cmd.append("-p-")
        cmd.append(target_archive)

        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in iter(proc.stdout.readline, ''):
            l = line.strip()
            if "%" in l or "Extracting" in l:
                yield f"data: {json.dumps({'status': 'progress', 'message': f'⏳ {l[:80]}' }}\\n\\n"
                await asyncio.sleep(0.04)
        proc.wait()

        if proc.returncode == 0:
            yield f"data: {json.dumps({'status': 'success', 'progress': 100, 'message': f'🎉 ✅ SUCCESS! All files extracted to MyDrive/{req.destination_folder}'})}\\n\\n"
        else:
            yield f"data: {json.dumps({'status': 'error', 'message': f'❌ Extraction failed with code {proc.returncode}'})}\\n\\n"
    return StreamingResponse(event_generator(), media_type="text/event-stream")
'''
with open('/content/server.py', 'w') as f:
    f.write(server_code)

# 5. Start background server
subprocess.Popen(["pkill", "-f", "uvicorn"], check=False)
time.sleep(1)
subprocess.Popen(["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"], cwd="/content")
time.sleep(2)

# 6. Start Cloudflare Tunnel
subprocess.Popen(["pkill", "-f", "cloudflared"], check=False)
tunnel_proc = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8000"],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

print("⏳ Generating secure link to your Android phone...")
tunnel_url = None
for _ in range(80):
    line = tunnel_proc.stdout.readline()
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        tunnel_url = m.group(0)
        break
    time.sleep(0.1)

if tunnel_url:
    print("\n" + "=" * 65)
    print("🎉 CLOUD WORKER IS ONLINE & CONNECTED TO GOOGLE DRIVE!")
    print("=" * 65)
    print("\n📱 COPY & PASTE THIS URL INTO YOUR ANDROID APP:")
    print(f"\n👉  {tunnel_url}  👈\n")
    print("=" * 65)
    print("Keep this notebook running. Whenever you tap 'START CLOUD EXTRACTION' on your phone, files will be extracted directly here!")
else:
    print("❌ Could not detect Cloudflare URL. Please check connection.")
